In [1]:
pdf_path = '2025舞光LED21st(單頁水印可搜尋).pdf'

In [1]:
import os
import re
import uuid
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional

# ==========================================
# 1. SETUP & INITIALIZATION
# ==========================================
# Make sure OPENAI_API_KEY is in your environment variables
openai_client = OpenAI()

# Use Chroma's official OpenAI embedding wrapper
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ.get("OPENAI_API_KEY"),
    model_name="text-embedding-3-small"
)

# Initialize Chroma natively
chroma_client = chromadb.PersistentClient(path="./dancelight_vanilla_db")
collection = chroma_client.get_or_create_collection(
    name="dancelight_leds",
    embedding_function=openai_ef
)

# ==========================================
# 2. THE CUSTOM MARKDOWN CHUNKER
# ==========================================
def chunk_markdown_with_headers(markdown_text):
    current_metadata = {"Category": None, "Series": None, "Product_Name": None}
    chunks = []
    current_chunk_text = []
    header_pattern = re.compile(r"^(#{1,3})\s+(.*)$")
    
    for line in markdown_text.split('\n'):
        match = header_pattern.match(line)
        if match:
            # Save the previous chunk if it has text
            if current_chunk_text:
                chunks.append({
                    "id": str(uuid.uuid4()),
                    "text": "\n".join(current_chunk_text).strip(),
                    "metadata": {k: v for k, v in current_metadata.items() if v is not None}
                })
                current_chunk_text = [] # Reset text for the new section
                
            # Update our metadata tracker with the new header
            level = len(match.group(1)) 
            header_text = match.group(2).strip()
            
            if level == 1:
                current_metadata["Category"] = header_text
                current_metadata["Series"] = None
                current_metadata["Product_Name"] = None
            elif level == 2:
                current_metadata["Series"] = header_text
                current_metadata["Product_Name"] = None
            elif level == 3:
                current_metadata["Product_Name"] = header_text
        else:
            if line.strip(): 
                current_chunk_text.append(line)
                
    # Save the very last chunk
    if current_chunk_text:
        chunks.append({
            "id": str(uuid.uuid4()),
            "text": "\n".join(current_chunk_text).strip(),
            "metadata": {k: v for k, v in current_metadata.items() if v is not None}
        })
    return chunks

def ingest_documents(markdown_filepath):
    print(f"Reading {markdown_filepath}...")
    with open(markdown_filepath, "r", encoding="utf-8") as f:
        full_text = f.read()
        
    processed_chunks = chunk_markdown_with_headers(full_text)
    
    if not processed_chunks:
        print("Error: No chunks were created. Check your markdown formatting.")
        return
        
    ids = [chunk["id"] for chunk in processed_chunks]
    documents = [chunk["text"] for chunk in processed_chunks]
    metadatas = [chunk["metadata"] for chunk in processed_chunks]
    
    collection.upsert(
        documents=documents,
        ids=ids,
        metadatas=metadatas
    )
    print(f"✅ Ingested {len(processed_chunks)} hierarchically aware chunks into ChromaDB.")

# ==========================================
# 3. METADATA EXTRACTION (Replacing SelfQueryRetriever)
# ==========================================
# We define exactly what we want OpenAI to extract, matching our chunker's metadata!
class LEDFilter(BaseModel):
    Category: Optional[str] = Field(default=None, description="The top-level category (e.g., Downlight, Strip light)")
    Series: Optional[str] = Field(default=None, description="The product series name")
    Product_Name: Optional[str] = Field(default=None, description="The specific product name")

def extract_filters(user_query: str) -> dict:
    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Extract the LED lighting attributes the user is looking for. Do not guess; if an attribute is not explicitly mentioned, leave it out."},
            {"role": "user", "content": user_query}
        ],
        response_format=LEDFilter
    )
    return completion.choices[0].message.parsed.model_dump(exclude_none=True)

# ==========================================
# 4. THE RAG GENERATOR
# ==========================================
def ask_dancelight(user_query: str):
    # Step A: Ask GPT-4o to extract any hard filters
    filters = extract_filters(user_query)
    print(f"\n🔍 Extracted Metadata Filters: {filters}")
    
    # Step B: Construct the native Chroma DB query
    search_kwargs = {"query_texts": [user_query], "n_results": 3}
    
    if filters:
        # If there are multiple filters, Chroma requires the $and operator
        if len(filters) == 1:
            search_kwargs["where"] = filters
        else:
            search_kwargs["where"] = {"$and": [{k: v} for k, v in filters.items()]}
            
    # Step C: Retrieve the matching documents
    results = collection.query(**search_kwargs)
    
    if not results["documents"] or not results["documents"][0]:
        return "No matching LED products found in the database."
        
    context = "\n---\n".join(results["documents"][0])
    
    # Step D: Final Answer Generation
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful product assistant for DanceLight LED products. Answer using ONLY the provided context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {user_query}"}
        ]
    )
    return response.choices[0].message.content

# ==========================================
# 5. TEST THE SYSTEM
# ==========================================
if __name__ == "__main__":
    # 1. Run this ONCE to fill your database. Make sure the filename matches your Docling output!
    ingest_documents("dancelight_knowledge_base.md") 
    
    # 2. Test the query pipeline
    user_question = "Do you have any Downlights in the Pro Series?"
    answer = ask_dancelight(user_question)
    print(f"\n🤖 Answer:\n{answer}")

Reading dancelight_knowledge_base.md...


ValueError: Expected metadata to be a non-empty dict, got 0 metadata attributes in upsert.

In [3]:
import os
import json
import uuid
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional

# ==========================================
# 1. SETUP & INITIALIZATION
# ==========================================
# Ensure OPENAI_API_KEY is set in your environment variables
openai_client = OpenAI()

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ.get("OPENAI_API_KEY"),
    model_name="text-embedding-3-small"
)

# Using a new folder for the JSON-based database
chroma_client = chromadb.PersistentClient(path="./dancelight_json_db")
collection = chroma_client.get_or_create_collection(
    name="dancelight_leds_advanced",
    embedding_function=openai_ef
)

# ==========================================
# 2. ADVANCED METADATA FILTER MODEL
# ==========================================
class LEDFilter(BaseModel):
    model_number: Optional[str] = Field(default=None, description="The exact model number (e.g., D-UTMTSPL12NRT)")
    series: Optional[str] = Field(default=None, description="The product series name")
    watt: Optional[float] = Field(default=None, description="Exact power consumption in watts")
    # Added "max_watt" and "max_price" to handle range questions!
    max_watt: Optional[float] = Field(default=None, description="Maximum power consumption if the user says 'under', 'less than', or 'up to' X watts")
    cct: Optional[float] = Field(default=None, description="Color temperature in Kelvin (e.g., 3000, 4000)")
    beam: Optional[float] = Field(default=None, description="Beam angle in degrees")
    lumen: Optional[float] = Field(default=None, description="Brightness in lumens")
    cri: Optional[float] = Field(default=None, description="Color Rendering Index (e.g., 90.0)")
    ip: Optional[str] = Field(default=None, description="IP rating for water/dust resistance (e.g., IP65)")
    voltage: Optional[str] = Field(default=None, description="Input voltage (e.g., DC24V, AC220V)")
    price: Optional[float] = Field(default=None, description="Exact product price")
    max_price: Optional[float] = Field(default=None, description="Maximum price if user says 'under', 'cheaper than', etc.")

def extract_filters(user_query: str) -> dict:
    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Extract LED lighting attributes from the query. Use max_watt or max_price if the user asks for upper limits. Do not guess."},
            {"role": "user", "content": user_query}
        ],
        response_format=LEDFilter
    )
    return completion.choices[0].message.parsed.model_dump(exclude_none=True)

# ==========================================
# 3. JSON INGESTION SCRIPT (Replaces Markdown Chunker)
# ==========================================
def ingest_json_data(json_filepath: str):
    print(f"Reading {json_filepath}...")
    with open(json_filepath, "r", encoding="utf-8") as f:
        products = json.load(f)
        
    ids = []
    documents = []
    metadatas = []
    
    for prod in products:
        doc_id = str(uuid.uuid4())
        
        # 1. Map and Clean the metadata
        # (We rename 'model' to 'model_number' just to avoid database keyword conflicts)
        raw_meta = {
            "model_number": prod.get("model"),
            "watt": prod.get("watt"),
            "cct": prod.get("cct"),
            "beam": prod.get("beam"),
            "lumen": prod.get("lumen"),
            "cri": prod.get("cri"),
            "ip": prod.get("ip"),
            "voltage": prod.get("voltage"),
            "price": prod.get("price"),
            "price_from": prod.get("price_from"),
            "series": prod.get("series")
        }
        
        # --- THE FIX: Strip out None and "" values entirely ---
        clean_meta = {k: v for k, v in raw_meta.items() if v not in [None, ""]}
        
        # Failsafe so ChromaDB never gets an empty dictionary
        if not clean_meta:
            clean_meta = {"type": "General LED"}
            
        # 2. Create a rich text paragraph for the AI to read and answer from
        text_lines = [f"Series: {prod.get('series', 'Unknown')}"]
        text_lines.append(f"Model: {prod.get('model', 'Unknown')}")
        if prod.get('watt'): text_lines.append(f"Wattage: {prod.get('watt')}W")
        if prod.get('cct'): text_lines.append(f"Color Temp: {prod.get('cct')}K")
        if prod.get('lumen'): text_lines.append(f"Lumens: {prod.get('lumen')}lm")
        if prod.get('price'): text_lines.append(f"Price: ${prod.get('price')}")
        if prod.get('ip'): text_lines.append(f"IP Rating: {prod.get('ip')}")
        
        document_text = "\n".join(text_lines)
        
        ids.append(doc_id)
        documents.append(document_text)
        metadatas.append(clean_meta)

    collection.upsert(
        documents=documents,
        ids=ids,
        metadatas=metadatas
    )
    print(f"✅ Successfully ingested {len(products)} products into ChromaDB.")

# ==========================================
# 4. THE RAG GENERATOR (With Advanced Queries)
# ==========================================
def build_chroma_where_clause(filters: dict) -> Optional[dict]:
    """Converts the extracted filter dictionary into ChromaDB's query language."""
    conditions = []
    
    for key, value in filters.items():
        if key == "max_price":
            # $lte = Less Than or Equal To
            conditions.append({"price": {"$lte": value}}) 
        elif key == "max_watt":
            conditions.append({"watt": {"$lte": value}})
        else:
            # $eq = Exact Match
            conditions.append({key: {"$eq": value}}) 
            
    if not conditions:
        return None
    if len(conditions) == 1:
        return conditions[0]
    return {"$and": conditions} # Wrap multiple filters in an $and operator

def ask_dancelight(user_query: str):
    filters = extract_filters(user_query)
    print(f"\n🔍 Extracted Filters: {filters}")
    
    where_clause = build_chroma_where_clause(filters)
    search_kwargs = {"query_texts": [user_query], "n_results": 3}
    
    if where_clause:
        print(f"⚙️ DB Query Logic: {where_clause}")
        search_kwargs["where"] = where_clause
        
    try:
        results = collection.query(**search_kwargs)
    except Exception as e:
        return "Sorry, I couldn't search the database for those specific parameters right now."
        
    if not results["documents"] or not results["documents"][0]:
        return "No matching LED products found matching those exact specs."
        
    context = "\n---\n".join(results["documents"][0])
    
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful product assistant for DanceLight LED products. Answer using ONLY the provided context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {user_query}"}
        ]
    )
    return response.choices[0].message.content
ask_dancelight("我要浴室裡的燈 ")
# # ==========================================
# # 5. EXECUTION & TESTING
# # ==========================================
# if __name__ == "__main__":
#     # 1. Automatically generate a dummy JSON file to test the script
#     dummy_data = [
#         {
#             "model": "D-UTMTSPL12NRT", "watt": 12.0, "cct": 4000.0, "beam": 30.0, 
#             "lumen": 1010.0, "cri": 90.0, "ip": "", "voltage": "DC24V", 
#             "price": 7500.0, "price_from": "fuzzy", "series": "拉斐爾超薄磁吸"
#         },
#         {
#             "model": "D-CHEAPDOWNLIGHT", "watt": 8.0, "cct": 3000.0, "beam": 120.0, 
#             "lumen": 800.0, "cri": 80.0, "ip": "IP44", "voltage": "AC220V", 
#             "price": 1500.0, "price_from": "manual", "series": "Basic Series"
#         }
#     ]
#     with open("dancelight_products.json", "w", encoding="utf-8") as f:
#         json.dump(dummy_data, f, ensure_ascii=False, indent=2)

#     # 2. Run the ingestion (Notice it will safely ignore the empty 'ip' string for the first product)
#     ingest_json_data("dancelight_products.json")
    
#     # 3. Test the exact match logic
#     print("\n--- Test 1: Exact Match ---")
#     answer1 = ask_dancelight("Tell me about the D-UTMTSPL12NRT.")
#     print(f"🤖 Answer:\n{answer1}")
    
#     # 4. Test the advanced filtering logic
#     print("\n--- Test 2: Advanced Filtering (Under 10W) ---")
#     answer2 = ask_dancelight("Do you have any lights under 10 watts?")
#     print(f"🤖 Answer:\n{answer2}")


🔍 Extracted Filters: {}


'基於您對浴室照明的需求，可以考慮使用防護等級足夠的IP44產品。從提供的資訊中，D-CHEAPDOWNLIGHT（IP44）可能適合您的需求。'